# OCR a ticker's filings — VCB

⚠️ **A FORK OF `RUN__pdf_ocr_control.ipynb`, AND EVERY CELL BUT THE PARAMETERS ONE IS THAT
NOTEBOOK VERBATIM.** Re-cut from it on 2026-09-02, because the previous fork had drifted: it
predated the two-pass merge section and the §3 gap view, and nothing in it said so. If you change
the machinery, change the BASE and re-cut — never here.

## 1 · Parameters — the only cell you edit

In [ ]:
# ── PARAMETERS — the only cell you edit ───────────────────────────────
ENVIRONMENT = "LOCAL"        # "LOCAL" = parse here | "KAGGLE" = ship it to a T4
EXCHANGE    = "HOSE"         # HOSE | HNX | UPCOM
SYMBOL      = "VCB"          # ticker, as CafeF files it

# WHICH QUARTERS — YYYY-QQ. "2026-Q4" and the zero-padded "2026-04" are the same quarter.
#   []  or  None  ->  EVERY quarter this ticker files   (⚠️ 70 documents, hours)
#   "OUTSTANDING" ->  exactly the quarters §3 finds still `missing` AND still winnable.
#                     Resolved from the three statement CSVs and the PDF index, printed
#                     before anything is spent, and it RAISES rather than falling through
#                     to "every quarter" when there is nothing left to do.
# ⚠️ The repo-native "Q3-2014" is REFUSED rather than quietly accepted: a typo has to report
#    itself as a typo, not as a quarter CafeF does not file.
# ⚠️ **THE SENTINEL, AND IT IS THE POINT OF THIS FORK.** Naming quarters by hand is how the
#    request that started this arrived asking for Q3-2009, which had read `pdf` in all three
#    statements since it was first parsed, while the two cells that were actually missing went
#    unnamed. §3 prints the gap; this reads it.
# ⚠️ **AS OF 2026-09-02 VCB IS 70 OF 70 COMPLETE, so this RAISES — and that is the designed
#    answer, not a fault.** An empty result must not fall through to "every quarter": `plan()`
#    reads an empty `quarters` list as ALL, so a "nothing left to do" answer would otherwise open
#    70 filings. To re-parse something anyway, name the quarters here and set OVERWRITE.
QUARTERS = "OUTSTANDING"

# ⚠️ What to do about a quarter ALREADY on disk:
#   False -> FILL THE GAPS. One reading `pdf` in all three statements is dropped before any OCR
#            (and before it is uploaded); a figure that DIFFERS is never written over it.
#   True  -> re-parse every selected quarter and let the result replace what disk holds.
#   ⚠️ To replace ONE wrong row use REPAIR below, never this — see the note there.
# ⚠️ FALSE, and with the sentinel it is the only coherent setting: §3 hands this run the
#    quarters that are NOT complete, so there is nothing here to overwrite.
OVERWRITE = False

# UPSERT the accepted statements into raw_data/.../statements/*.csv, through `pdf_ocr_merge`:
# it BACKS THE THREE CSVs UP FIRST, prints every changed cell, and refuses four things it
# cannot judge — a cumulative income statement whose missing quarters WERE filed, a statement
# whose `sane` band was empty, a figure that DIFFERS from a good `pdf` row, and ⚠️ a document any
# of whose layers RAISED (`VCR-1`: an exception measures the MACHINE, not the filing, so
# whatever won the cascade won by default).
#   LOCAL  -> one quarter at a time, as each finishes. This is the interruption guarantee.
#   KAGGLE -> once, after the pull. A kernel has no path to this disk.
# ⚠️ **OFF, AND `NST-2` IS WHY.** VCB Q1-2009's balance sheet ACCEPTS since `NST-1`
#    (2026-09-02), at `onnx@400+loose`, and the reading is **SLID BY ONE ROW**: measured against
#    the Q2-2009 filing's own 31/12/2008 comparative column, which both filings print, **37 of 57
#    figures sit under a different label**. Its three grand totals land correctly
#    (205,501,553,372,238 + 14,884,328,899,019 + 107,573,244,033 = 220,493,455,515,290, exact),
#    so `reconcile` and `sane` BOTH PASS and the automatic merge writes it without a word —
#    which is what happened, on the owner's decision. **A statement that clears both gates is not
#    a statement that is right**, and this knob is what puts a person between the two.
#    ⚠️ Turn it on only for a target whose §7 verdicts you have read.
MERGE_INTO_CSV = False

# ⚠️ BOOTSTRAP A TICKER THAT HAS NO STATEMENT CSV YET — and it lifts a real guard (`BND-1`).
#   True  -> write a statement whose `sane` band was EMPTY. The ONLY way a new ticker starts.
#   False -> keep the guard. Correct for a ticker that already has history on disk.
# ⚠️ FALSE: VCB carries 70 quarters, and `seed_history` finds a real band for both outstanding
#    ones (Q4-2008's balance sheet sits before them). Turning it on here would lift a guard that
#    is working.
FORCE_EMPTY_BAND = False

TEMPLATE     = None      # None = RESOLVE it (templates.csv, then CafeF's fingerprint). ⚠️ Never defaulted to "bank".
ALLOW_PARENT = False     # fall back to the STANDALONE filing where no consolidated one exists
PERIODS      = None      # the repo-native form, e.g. ["Q3-2014"]. Optional, and INTERSECTS with QUARTERS.
LAYERS       = None      # None = the full cascade, in cascade order
COMPARE      = True      # score every parsed cell against the statement CSV already on disk
NOTES        = ""        # free text into the run folder; blank writes a sensible default

# ── THE MERGE — section 9, one period at a time, oldest first, and UNFORCED ───────────
# ⚠️ THIS IS THE LOCAL PER-QUARTER MERGE, GENERALISED TO ANY RUN FOLDER AND STRIPPED OF
#    `force_differs`. Merging period by period is not a style choice: `merge_run` plans against
#    disk and writes afterwards, so a span recorded for one quarter reaches the NEXT quarter's
#    planner only in the following call.
# ⚠️ NOT NEEDED FOR THIS TARGET and left on anyway — the outstanding cell is a BALANCE SHEET,
#    which carries no span and de-cumulates nothing, so one pass and two write the same rows.
#    ⚠️ **THIS SECTION IS THE ONLY WRITE PATH LEFT IN THIS FORK**, and `MERGE_APPLY` is False:
#    read the §7 verdicts and the plan it prints before setting it.
MERGE_TWO_PASS = True
# ⚠️ WHICH STATEMENTS PASS 2 MAY WRITE. None = all three.
MERGE_REPORTS = None
MERGE_APPLY   = False    # False = PLAN ONLY, nothing is written. True once you have read it.

# ⚠️ REPAIR — REPLACE A `pdf` ROW THAT IS ALREADY ON DISK AND WRONG. Name the exact
#    (quarter, statement) pairs; anything not named keeps the DIFFERS refusal. The quarter is
#    the REPO-NATIVE form here:   REPAIR = [("Q3-2009", "income_statement")]
# ⚠️ `OVERWRITE = True` IS THE WRONG TOOL FOR THIS, AND THE REASON IS MEASURED. It lifts DIFFERS
#    for every statement of every quarter in the run — and a `pdf_ocr_job` run is NOT the run
#    that wrote those rows: its `sane` band is rebuilt from disk where a full `build()`
#    accumulates one as it goes, so the two escalate DIFFERENTLY and the seeded run can win on
#    an EARLIER, POORER layer. On ACB 2026-08-30 it would have replaced a 33-item balance sheet
#    with a 19-item one while repairing another statement, reporting only "DIFFERS in N columns".
# ⚠️ Read the DIFFERS report in section 7 first, and decide against the FILING (a printed
#    subtotal, the next quarter's comparative column) — never by preferring the newer run.
REPAIR = []
REPAIR_APPLY = False     # False = print the plan and change nothing. True once you agree.

EXECUTE  = True          # False = resolve and print the plan, spend nothing
REHEARSE = True          # KAGGLE only: the worker side, locally, no quota (~60 s)


## 2 · Setup — validate the parameters, find the repo

In [ ]:
# ── SETUP — validate the parameters and find the repo ─────────────────────────
# ⚠️ Checked HERE, before a payload is built or a page is rendered: every one of these is a
# mistake that would otherwise surface hours later, or as a spent Kaggle round trip.
import os
import sys
from pathlib import Path

ENVIRONMENT = str(ENVIRONMENT).upper()
EXCHANGE = str(EXCHANGE).upper()
SYMBOL = str(SYMBOL).upper()
if ENVIRONMENT not in ("LOCAL", "KAGGLE"):
    raise ValueError(f"ENVIRONMENT must be 'LOCAL' or 'KAGGLE', not {ENVIRONMENT!r}")
if EXCHANGE not in ("HOSE", "HNX", "UPCOM"):
    raise ValueError(f"EXCHANGE must be HOSE, HNX or UPCOM, not {EXCHANGE!r}")

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "kaggle_gpu").is_dir()), None)
if REPO is None:
    raise RuntimeError(f"no src/kaggle_gpu at or above {Path.cwd()} — open this notebook "
                       f"from inside the repo.")
# ⚠️ `kgpu` stages the payload and talks to the Kaggle client relative to the CWD, so the
# notebook anchors itself the way a shell would. LOCAL does not need it and gets it anyway:
# one behaviour, printed, beats two that differ by a mode.
os.chdir(REPO / "src" / "kaggle_gpu")
for _p in (REPO / "src", REPO / "src" / "kaggle_gpu"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# ⚠️ A LONG-LIVED KERNEL PINS THE REPO TO THE COMMIT IT FIRST IMPORTED — `import` is a no-op
# once a module is in `sys.modules`, so re-running this notebook after the repo moves underneath
# it runs the OLD code. The loud form is an AttributeError; ⚠️ the silent form is an OCR run
# executing a previous commit's parser while `metadata.json` records HEAD's hash — a run folder
# that names code it did not run. So the repo's OWN packages are dropped here and re-imported
# from disk on every pass; third-party ones (torch, onnxruntime) are left alone, they do not
# move. ⚠️ It re-imports, so run this notebook TOP TO BOTTOM.
_OURS = ("kgpu", "utils", "web_scraper")
_RELOADED = [_n for _n in list(sys.modules) if _n.split(".")[0] in _OURS]
for _n in _RELOADED:
    del sys.modules[_n]

from utils import progress                          # noqa: E402
from web_scraper import pdf_ocr_job as job          # noqa: E402

# ⚠️ FOLDED ONCE, HERE. "2026-04" and "2026-Q4" are one quarter, and the job name, the payload
# directory and the Kaggle kernel slug are all derived from this list — two spellings that
# reached those would be two runs racing for one slug. `None` means every quarter.
# ⚠️ THE THIRD FORM IS A SENTINEL AND IS RESOLVED IN §3, NOT HERE — `canonical_quarters`
# refuses a bare string, and §3 is where the statement CSVs and the PDF index are read.
OUTSTANDING_ONLY = isinstance(QUARTERS, str) and QUARTERS.strip().upper() == "OUTSTANDING"
QUARTERS = None if OUTSTANDING_ONLY else job.canonical_quarters(QUARTERS)

# The task label every progress line carries. ONE string, built once: LOCAL replaces it per
# document (`doc 2/3 HOSE_TCB Q3-2013`), KAGGLE keeps it for all six steps.
# ⚠️ Rebuilt in §3 when the sentinel resolves: a label reading "all quarters" over a
# three-quarter run is a progress line lying about its own denominator.
LABEL = f"{EXCHANGE}_{SYMBOL} " + (" ".join(QUARTERS) if QUARTERS else "all quarters")

print(f"environment : {ENVIRONMENT}")
print(f"ticker      : {EXCHANGE}_{SYMBOL}")
print("quarters    : " + ("OUTSTANDING — resolved in §3 from what is on disk"
                          if OUTSTANDING_ONLY else
                          f"{QUARTERS or 'ALL — every quarter this ticker files'}"))
print(f"overwrite   : {OVERWRITE}"
      + ("" if OVERWRITE else "   (quarters already `pdf` in all three are skipped)"))
print(f"upsert csv  : {MERGE_INTO_CSV}"
      + ("   per quarter, as each finishes" if MERGE_INTO_CSV and ENVIRONMENT == "LOCAL"
         else "   after the pull" if MERGE_INTO_CSV else ""))
print(f"bootstrap   : {FORCE_EMPTY_BAND}"
      + ("   an EMPTY `sane` band is written anyway — the only way a new ticker "
         "starts" if FORCE_EMPTY_BAND else "   an EMPTY `sane` band is REFUSED"))
print(f"repo        : {REPO}")
print(f"cwd         : {Path.cwd()}")
print(f"code        : {REPO / 'src'}"
      + (f"   ({len(_RELOADED)} cached module(s) dropped, re-imported from disk)"
         if _RELOADED else "   (first import in this kernel)"))
# ⚠️ The percentage is a POSITION IN THE PLAN and not a fraction of the time left — a filing
# accepted at layer 1 of 47 costs ~1 min and one that defeats the cascade cost 33. Said here,
# once, because it is on every line below it.
print(f"log shape   : {progress.format_line(0.337, 'task', 'sub-task', 'detail')}"
      f"   ← overall %, a position in the plan")


## 3 · What is left — the gap on disk, and what a re-run cannot change

In [ ]:
# ── WHAT IS LEFT — the gap on disk, and what a re-run cannot change ───────────
# ⚠️ THE QUESTION THIS ANSWERS IS THE ONE THAT DECIDES `QUARTERS`, and until 2026-09-02 the
# notebook could not answer it: you had to know which (quarter, statement) cells of this ticker
# still read `missing`, and the only way to find out was an ad-hoc script over the three CSVs.
# A request that named the wrong quarters was therefore indistinguishable from one that named
# the right ones until an hour of GPU had been spent on it.
#
# ⚠️ IT IS NOT A SECOND RULE. The quarters come from `documents()` through `job.plan()` — the
# same call the run makes — and "already done" is `job.parsed_reports()`, which is `pdf` and
# nothing else (rule 24 leaves no third answer: a `cafef` row is a transcription, and a
# `missing` row is a quarter nothing was written for).
#
# ⚠️ `use_data_root()` FIRST, AND IT IS LOAD-BEARING (`CWD-1`). `fin.STATEMENTS_DIR` is a
# RELATIVE default read at call time, and §2 has just `os.chdir`-ed into `src/kaggle_gpu` — so
# without this every quarter reads `absent`, which is a legitimate state for a ticker being
# bootstrapped and therefore looks like nothing is wrong.
from web_scraper import cafef_financials as fin      # noqa: E402

job.use_data_root(REPO / "raw_data" / "cafef")
_builder = fin.FinancialsBuilder(logger=None)

# ⚠️ RESOLVED, NEVER DEFAULTED — and how it resolved is printed, because "read off
# templates.csv" and "fingerprinted over the network" are not the same claim (`TPX-1`).
TEMPLATE_HOW = "given"
if TEMPLATE is None:
    TEMPLATE, TEMPLATE_HOW = job.resolve_template(_builder, SYMBOL)

_filed = job.plan(_builder, EXCHANGE, SYMBOL, allow_parent=ALLOW_PARENT, template=TEMPLATE)
SETTLED = job.settled_absences(REPO / "reports" / "pdf_ocr", EXCHANGE, SYMBOL)

# ⚠️ `SETTLED` is keyed `YYYY-QQ` — the sortable form QUARTERS is written in — while a task
# carries the repo-native `QQ-YYYY`. Comparing the two spellings directly matches NOTHING and
# looks exactly like "nothing is settled"; `as_quarter` is the converter.
GAPS, OUTSTANDING, _n_settled = {}, [], 0
for _task in _filed:
    _quarter = job.as_quarter(_task.period)
    _done = set(job.parsed_reports(_builder, _task))
    _gap = [r for r in job.REPORTS if r not in _done]
    if not _gap:
        continue
    _settled_here = SETTLED.get(_quarter, {})
    GAPS[_quarter] = {r: _settled_here.get(r) for r in _gap}
    _n_settled += sum(1 for r in _gap if r in _settled_here)
    if any(r not in _settled_here for r in _gap):
        OUTSTANDING.append(_quarter)

print(f"{EXCHANGE}_{SYMBOL}   template {TEMPLATE} ({TEMPLATE_HOW})   "
      f"{len(_filed)} quarter(s) filed, {len(_filed) - len(GAPS)} complete")
print("")
# ⚠️ NO FILINGS AND NOTHING OUTSTANDING PRINT THE SAME LINE OTHERWISE, and they are opposite
# answers: one says the ticker is done, the other that nothing was ever measured (§5 rule 2).
if not _filed:
    print("  ⚠️ this ticker files NO document `documents()` will open — an absent PDF index, or")
    print(f"     everything before FINANCIALS_PERIOD_MIN. Nothing here says the ticker is done.")
elif not GAPS:
    print("  every filed quarter reads `pdf` in all three statements. Nothing is outstanding.")
else:
    for _quarter in sorted(GAPS):
        for _report, _run in sorted(GAPS[_quarter].items()):
            print(f"  {_quarter:<9} {_report:<18} "
                  + (f"SETTLED — the filing contains no such statement  ({_run})"
                     if _run else "open — a re-run could still win it"))
    print("")
    print(f"  {sum(len(v) for v in GAPS.values())} outstanding cell(s), of which "
          f"{_n_settled} are SETTLED and {len(OUTSTANDING)} quarter(s) have an OPEN one.")

# ⚠️ A SETTLED CELL IS `missing` FOREVER, and re-running it costs the full cascade to return the
# same word. ACB's Q2-2009 and Q3-2009 cash flows were put through all 50 layers FOUR times on
# 2026-08-30 before anything recorded why: both filings are three-page `BÁO CÁO TÀI CHÍNH TÓM
# TẮT` forms (Mẫu CBTT-03) with no cash flow statement in them at all.
# ⚠️ AND AN EMPTY `SETTLED` IS SILENCE, NOT A CLEAN BILL: a run older than artefact schema v4
# recorded no reason, so a cell reading "open" here may still be unwinnable and merely unmeasured
# (§5 rule 2).
if _n_settled:
    print("")
    print("  `missing` is the correct and PERMANENT answer for the SETTLED rows (§5 rule 24).")
    print("  Keep such a quarter only to re-parse the OTHER statements of the same filing.")

# ⚠️ THE SENTINEL, RESOLVED HERE AND NOWHERE ELSE. An empty result RAISES rather than falling
# through: `plan()` reads an empty `quarters` as "every quarter this ticker files", so the one
# thing a "nothing is left to do" answer must not do is silently open 70 filings.
if OUTSTANDING_ONLY:
    if not OUTSTANDING:
        raise RuntimeError(
            f'QUARTERS = "OUTSTANDING" resolved to nothing for {EXCHANGE}_{SYMBOL}: every '
            f"filed quarter either reads `pdf` in all three statements or is SETTLED. Name "
            f"the quarters explicitly if you meant to re-parse something anyway.")
    QUARTERS = OUTSTANDING
    LABEL = f"{EXCHANGE}_{SYMBOL} " + " ".join(QUARTERS)
    print("")
    print(f'  QUARTERS = "OUTSTANDING" -> {QUARTERS}')


## 4 · The plan — what would run, before anything is spent

In [ ]:
# ── THE JOB — resolved and printed, before anything is spent ──────────────────
# ⚠️ Both branches end at the SAME object. `pdf_ocr.job()` writes a `JobSpec`'s fields into the
# worker notebook's parameter cell, and the worker builds the JobSpec from them — so a LOCAL run
# and a KAGGLE run of the same parameters are one procedure on two machines, not two. What
# differs is the stack, and every run records its `stack_fingerprint`.
SPEC = CFG = PREPARED = None

if ENVIRONMENT == "LOCAL":
    SPEC = job.JobSpec(
        exchange=EXCHANGE, symbol=SYMBOL, periods=PERIODS, quarters=QUARTERS,
        allow_parent=ALLOW_PARENT, overwrite=OVERWRITE, template=TEMPLATE, layers=LAYERS,
        compare_with_disk=COMPARE, merge_into_csv=MERGE_INTO_CSV,
        force_empty_band=FORCE_EMPTY_BAND,
        notes=NOTES or f"ENVIRONMENT=LOCAL overwrite={OVERWRITE}",
    )
    # ⚠️ `prepare()` resolves the data root, the models, the TEMPLATE and the document list and
    # RAISES on any of them — no OCR, no PDF. It also raises, in as many words, when every
    # quarter you asked for is already parsed and OVERWRITE is False.
    PREPARED = SPEC.prepare()
    print("\n".join(PREPARED.describe()))
    print()
    for _t in PREPARED.tasks:
        print(f"  {_t.period:<8} {_t.file[:56]:<56} "
              f"{os.path.getsize(_t.path) / 1024 ** 2:>6.1f} MB"
              + ("  CUMULATIVE" if _t.cumulative else ""))
    # ⚠️ THE CEILING, BEFORE ANY OF IT IS SPENT. The bill is `pages x OCR passes`, and the 49
    # layers are only 7 passes — a layer that changes only the mapping or a gate re-maps a parse
    # the page cache already holds. Both numbers are free: `page_count` opens the PDF without
    # rendering a pixel, and the pass count is a property of the cascade.
    # ⚠️ It is a CEILING, loose in the honest direction: `scan` stops as soon as all three
    # statements are behind it (BID Q3-2011 reads 7 pages of 32) and the cascade stops at the
    # first layer that accepts. What it tells you is which filing would be dear IF something in
    # it cannot be read — that is the only case that pays it.
    import fitz                                       # noqa: E402
    from web_scraper.cafef_financials import ocr_key  # noqa: E402

    PASSES = len({ocr_key(_l) for _l in PREPARED.layers})
    PAGES = 0
    for _t in PREPARED.tasks:
        try:
            with fitz.open(_t.path) as _d:
                PAGES += _d.page_count
        except Exception as _e:                       # a damaged page tree is `scan`'s problem
            print(f"  ⚠️ could not count pages of {_t.file}: {_e}")
    print("")
    print(f"  ceiling      : {PAGES} page(s) x {PASSES} OCR pass(es) = "
          f"{PAGES * PASSES:,} page-reads at most")
    print(f"                 ~{PAGES * PASSES * 0.65 / 60:.0f} min at 0.65 s/page "
          f"(onnx@200 on this laptop; the 300/400 dpi passes cost more).")
    print("                 A filing accepted at layer 1 pays ONE pass over the pages "
          "up to its last")
    print("                 statement, which is the usual case — see the run log.")

    if PREPARED.template != "bank":
        print(f"\n⚠️ CRP-1: this is a `{PREPARED.template}` filing. `C_LIABILITIES` still "
              f"misses on corp,\n   so the balance sheet reconciles on the TRIVIAL "
              f"`assets == resources` — true by\n   construction on any page that reads both. "
              f"Nothing from a non-bank run may be\n   quoted as a fundamental yet.")
else:
    from kgpu import pdf_ocr, runner                 # noqa: E402

    CFG = pdf_ocr.job(
        SYMBOL, exchange=EXCHANGE, periods=PERIODS, quarters=QUARTERS,
        allow_parent=ALLOW_PARENT, overwrite=OVERWRITE, template=TEMPLATE, layers=LAYERS,
        compare=COMPARE, notes=NOTES, merge_statements=MERGE_INTO_CSV,
        # ⚠️ NOT a worker parameter. The worker cannot upsert — it writes /kaggle/working and
        # exits — so this is the PULL's knob, read by `runner.merge_statements` on this machine.
        force_empty_band=FORCE_EMPTY_BAND,
    )
    print("\n".join(pdf_ocr.describe(CFG)))
    print()
    # The filings this selects are the filings the WORKER will open: `plan()` runs HERE, so the
    # payload cannot diverge from the worker's own choice.
    runner.plan(CFG)


## 5 · Rehearse — KAGGLE only: the worker side, locally, no quota

In [ ]:
# ── STAGE + REHEARSE — KAGGLE only: the worker side, locally, no quota ────────
# ⚠️ THE PAYLOAD IS STAGED HERE, AND IT HAS TO BE: a rehearsal runs the worker against
# `.payload/<job>/`, so there is nothing to rehearse until that exists. `export` is local and
# free — it writes the zip, it does not upload; the RUN cell below re-exports and uploads, so
# nothing here commits you to anything.
# ⚠️ The rehearsal runs no OCR pass. What it proves is that the payload holds every input the
# parse reads, under BOTH of Kaggle's mount layouts, and it prints the magnitude band `sane`
# will get. AN EMPTY BAND IS THE WARNING TO STOP FOR: `sane` fails open without one, and that
# is the documented way a run writes a wrong figure (CLAUDE.md §6-2-octodecies).
if ENVIRONMENT == "KAGGLE" and REHEARSE:
    from kgpu import export                       # noqa: E402

    # Two steps, one line each, in the same shape the RUN cell prints — `capture()` re-emits
    # `export`'s and `rehearse`'s own output as the DETAIL of the step that produced it.
    DRESS = progress.Stages([("export", "stage payload", 1.0),
                             ("rehearse", "rehearse worker", 1.0)], label=LABEL)
    DRESS.begin("export", "local, no upload, no quota")
    with DRESS.capture():
        export.export(CFG)                        # -> .payload/<job>/  (no upload)
    DRESS.begin("rehearse", "both Kaggle mount layouts")
    with DRESS.capture():
        runner.rehearse(CFG)
    DRESS.done("rehearsed — nothing was spent")
else:
    print("skipped" if ENVIRONMENT == "KAGGLE" else "LOCAL — nothing to rehearse")


## 6 · Run

In [ ]:
# ── RUN ───────────────────────────────────────────────────────────────────────
# ⚠️ One line shape on both machines — ` 33.7% - <task> - <sub-task> - <detail>`, one formatter
# (`utils.progress`), so the two cannot drift. LOCAL the task is the DOCUMENT and the sub-task
# its position in the cascade; KAGGLE the task is the STEP of the round trip.
# ⚠️ THE OVERALL % IS A POSITION IN THE PLAN, NOT A FRACTION OF THE TIME. A filing accepted at
# its FIRST OCR pass is ~1 min and one that defeats all 24 of them was 33, so the number is a
# LOWER BOUND — a run finishes early, it does not stall at 99 %. On KAGGLE it stands still
# through `wait kernel` unless this exact job has completed once before: `kernels_status`
# reports QUEUED / RUNNING / COMPLETE and no fraction.
# ⚠️ Each document's JSON is written BEFORE the next starts, and LOCAL each quarter is upserted
# as it lands — a run that kept its results in memory would lose them to the first interrupt.
# ⚠️ Budget: ~1 min for a filing accepted at layer 1, 26-33 min for one that defeats the whole
# cascade, plus Kaggle's ~5 min QUEUE before anything starts.
FOLDER = EXIT = REPORT = None

if not EXECUTE:
    print("EXECUTE = False — the plan above is resolved and nothing was spent")
elif ENVIRONMENT == "LOCAL":
    # `job.run` prints the progress line itself and writes the SAME line into the run
    # folder's `run.log`, so what you read here is what a later reader gets.
    FOLDER = job.run(SPEC)
else:
    if MERGE_INTO_CSV:
        print("MERGE_INTO_CSV is on: accepted statements are upserted into\n"
              "    raw_data/.../statements/ after the pull, with a backup taken first\n"
              "    and every changed cell printed.")
    # `refresh_data=True` re-exports and re-uploads the payload every time — correct, because
    # the filter above may have changed since the last run of this job.
    REPORT = progress.Stages(runner.RUN_STAGES, label=LABEL)
    EXIT = runner.run(CFG, refresh_data=True, progress=REPORT)
    REPORT.note(f"exit {EXIT}   (0 = COMPLETE and pulled)")


## 7 · The result — verdicts from the run folder

In [ ]:
# ── READ THE RUN FOLDER ─────────────────────────────────────────────────
# ⚠️ Read back from disk rather than from anything in memory, so this measures what a later
# reader would actually get. `metadata.json` already carries the whole scorecard in `results`.
import json                                          # noqa: E402

PATTERN = f"*__{EXCHANGE.lower()}_{SYMBOL.lower()}__pdf_ocr"
FOLDERS = sorted((REPO / "reports" / "pdf_ocr").glob(PATTERN), key=lambda p: p.name)
LATEST = FOLDERS[-1] if FOLDERS else None
META = MERGE = None

if LATEST is None:
    print(f"no run folder matching {PATTERN}")
else:
    META = json.loads((LATEST / "metadata.json").read_text(encoding="utf-8"))
    inputs, ocr = META.get("inputs", {}), META.get("environment", {}).get("ocr", {})
    SCHEMA = META.get("schema_version", 1)
    print(LATEST.name)
    print(f"  commit       : {META.get('git_commit')}")
    # ⚠️ §5 rule 2 at the artefact: an older run folder carries none of the fields below, and
    # printing `None` for them would read as a VALUE rather than as "this run predates the
    # field". `schema_version` is what tells the two apart.
    V2 = SCHEMA >= 2
    OLDER = "— (schema v1: this run predates the field)"
    print(f"  filter       : quarters={inputs.get('quarters')}  "
          f"periods={inputs.get('periods')}  "
          f"overwrite={inputs.get('overwrite') if V2 else OLDER}")
    print(f"  skipped      : {inputs.get('skipped_already_parsed') if V2 else OLDER}")
    print(f"  template     : {inputs.get('template')}  ({inputs.get('template_how')})")
    # ⚠️ THE TWO OCR HALVES FAIL INDEPENDENTLY — detection is onnxruntime, recognition is torch
    # — so "the GPU was used" is two questions. `ORT-1` is a green run that was half on the CPU
    # because onnxruntime ADVERTISED a provider the session then could not create.
    print(f"  detection    : {(ocr.get('det_providers') or ['?'])[0]}"
          f"   (onnxruntime {ocr.get('onnxruntime')})")
    print(f"  recognition  : {ocr.get('recognizer_device')}")
    print(f"  stack        : {ocr.get('stack_fingerprint')}"
          + (f"   ⚠️ PIN VIOLATIONS: {ocr['pin_violations']}"
             if ocr.get("pin_violations") else ""))

    # ⚠️ THE UPSERT IS THE ONE FIELD THE PARSING PROCESS CANNOT KNOW. `metadata.json` is written
    # by whatever ran the OCR, and on KAGGLE that is a worker with no path to this disk — so
    # `merged_into_csv` read `false` on every Kaggle run ever, whatever the pull did (`MRG-1`).
    # Since schema v3 the merge writes its own outcome back, and an ABSENT block on an older
    # folder means "this run predates the field", never "nothing was written".
    MERGE = META.get("merge")
    if MERGE:
        print(f"  upserted     : {MERGE['statements_written']} statement(s) written, "
              f"{MERGE['statements_skipped']} refused"
              + (f"   backup={inputs.get('merge_backup')}"
                 if inputs.get("merge_backup") else ""))
        if MERGE["periods_written"]:
            got = MERGE["periods_written"]
            print(f"                 {len(got)} quarter(s): "
                  f"{', '.join(got[:8])}{' …' if len(got) > 8 else ''}")
    elif SCHEMA >= 3:
        print("  upserted     : ⚠️ NOTHING — no merge ran against this run folder.")
    else:
        print(f"  upserted     : — (schema v{SCHEMA} predates the `merge` block; "
              f"`inputs.merged_into_csv` says {inputs.get('merged_into_csv')}, "
              f"which on a KAGGLE run was always false)")

    print()
    print(f"  {'period':10} {'report':18} {'layer':30} {'items':>5}  {'status':8} verdict")
    for r in META.get("results", []):
        print(f"  {r['period']:10} {r['report']:18} {(r['layer'] or '—'):30} "
              f"{r['items']:>5}  {r['status']:8} {r['verdict']}")
    # ⚠️ `seconds` is the DOCUMENT's cost repeated on each of its three report rows, so it is
    # summed per PERIOD. A set would also collapse two documents that took the same time.
    PER_DOC = {r["period"]: r["seconds"] for r in META.get("results", [])}
    print(chr(10) + f"  parse: {sum(PER_DOC.values()) / 60:.1f} min over "
          f"{len(PER_DOC)} document(s)")


## 8 · Refused vs written — two questions, two places

In [ ]:
# ── WHAT WAS REFUSED, AND WHAT WAS WRITTEN ───────────────────────────────
#   the PARSE refused a statement   -> `run.log`, written by whatever ran the OCR
#   the MERGE refused a statement   -> the `merge` block, written by whatever ran the UPSERT
# ⚠️ ON KAGGLE THOSE ARE TWO MACHINES. A cell that greps the worker's `run.log` for
# `WRITE `/`skip ` finds nothing on a Kaggle run and, finding nothing, used to print "no
# refusals — every statement was accepted". That false success was printed over a run that
# wrote 0 of 201 accepted cells (HOSE_CTG, 2026-08-30).
# ⚠️ MATCH ON THE DETAIL, NOT ON THE START OF THE LINE: since 2026-08-30 every line reads
# ` xx.x% - task - sub-task - detail`, and `progress.detail_of` is the segment that used to BE
# the line.
if LATEST is not None:
    LOG = (LATEST / "run.log").read_text(encoding="utf-8", errors="replace")
    HITS = [ln for ln in LOG.splitlines()
            if "absent after" in ln or "reconcile:" in ln or "sane:" in ln]
    print("── the PARSE refused ────────────────────────────────────────")
    print(chr(10).join(HITS) if HITS else
          "  nothing — every statement the cascade opened was accepted")

    # ⚠️ NOT a refusal — a fact about the FILING. A page whose scan is turned reads as vertical
    # noise, and before 2026-08-30 that cost a whole statement in silence (BID Q3-2011's income
    # statement, `no such statement on any page of this filing`, 47 times).
    TURNED = [ln for ln in LOG.splitlines() if "text lines are vertical" in ln]
    if TURNED:
        print(chr(10) + "── pages the READ had to turn ──────────────────────────────")
        for ln in TURNED:
            print("  " + progress.detail_of(ln))

    print(chr(10) + "── the MERGE decided ───────────────────────────────────────")
    if MERGE:
        for ev in MERGE["events"]:
            for d in ev["decisions"]:
                mark = "WRITE " if d["action"] == "write" else "skip  "
                items = f"[{d['layer']}] {d['items']} items" if d["layer"] else ""
                print(f"  {mark} {d['period']:9} {d['report']:18} {items:32} {d['reason']}")
                # ⚠️ A CAVEAT ON A WRITE IS LOUDER THAN A REFUSAL, because a refusal stops and
                # a write does not. Two exist: a row written as 6 or 12 months because its
                # priors were never filed, and — since 2026-08-31 — a quarter DE-CUMULATED
                # here, which names the periods that were subtracted from it. Printed only
                # for a WRITE: refusal 1 sets the note before refusals 2-4 have had their
                # say, so beside `skip` it would contradict the line above it.
                if d.get("note") and d["action"] == "write":
                    print(f"           ⚠️  {d['note']}")
        print(chr(10) + f"  -> {MERGE['statements_written']} written, "
              f"{MERGE['statements_skipped']} refused")
        if not MERGE["statements_written"]:
            print("  ⚠️ NOTHING REACHED raw_data/. On a ticker with no CSV yet the "
                  "commonest reason is an")
            print("     EMPTY `sane` band — set FORCE_EMPTY_BAND = True and merge again. "
                  "It lifts ONE guard")
            print("     and no other, so screen the artefact before quoting anything "
                  "(`BND-1`).")
    else:
        # A LOCAL run before schema v3 wrote its merge decisions into `run.log` instead.
        OLD = [ln for ln in LOG.splitlines()
               if progress.detail_of(ln).startswith(
                   ("WRITE ", "skip ", "backup:", "written:"))]
        if OLD:
            print(chr(10).join(OLD))
        else:
            print("  ⚠️ NO MERGE RAN against this run folder — the statement CSVs "
                  "were not opened.")
            print("     KAGGLE: `python -m kgpu merge <job>` finishes it "
                  "(--force-empty-band for a")
            print("     ticker with no CSV yet).   LOCAL: set MERGE_INTO_CSV = True.")


## 9 · The merge — one period at a time, oldest first, and UNFORCED

In [ ]:
# ── THE MERGE — one period at a time, oldest first, and UNFORCED ──────────
# ⚠️ WHY NOT ONE CALL OVER THE FOLDER: `merge_run` runs `plan_merge` against disk FIRST and
# `_write` afterwards, so every decision in one call is taken against the SAME disk state.
# `_quarter_priors` reads Q3-2019's `months` from that state, and `pending` carries only the
# quarters the merge itself DE-CUMULATED — which Q3-2019 is not, it is an ordinary quarterly.
# So the span Q3-2019 records reaches Q4-2019's planner only in the NEXT call.
# ⚠️ AND NOTHING HERE LIFTS A REFUSAL. `force_differs` is not passed, so a Q3-2019 reading that
# disagrees with disk is refused exactly as it would be by default and the span stays blank —
# which is the honest outcome, not a failure of this cell.
from web_scraper import cafef_financials as fin       # noqa: E402
from web_scraper import pdf_ocr_merge                 # noqa: E402

if not MERGE_TWO_PASS:
    print("MERGE_TWO_PASS = False — nothing was merged.")
elif LATEST is None or META is None:
    print("no run folder to merge — run the cells above first.")
else:
    # The periods THIS run folder holds, oldest first. Taken from the artefact rather than from
    # QUARTERS, because the plan may have dropped one and a merge must follow what was parsed.
    ORDER = sorted({r["period"] for r in META.get("results", [])}, key=fin._period_key)
    HOW = "APPLY" if MERGE_APPLY else "PLAN"
    print(f"{HOW} — {len(ORDER)} period(s) from {LATEST.name}, oldest first")
    print(f"       force_differs=False   force_empty_band={FORCE_EMPTY_BAND}")
    print(f"       reports={MERGE_REPORTS or 'all three'}"
          + ("   \u26a0\ufe0f the income statement is HELD BACK \u2014 see \u00a70's STOP"
             if MERGE_REPORTS and "income_statement" not in MERGE_REPORTS else ""))
    print()

    BACKED_UP = False
    for _i, _period in enumerate(ORDER, 1):
        print(f"── pass {_i}/{len(ORDER)}  {_period} " + "─" * 42)
        # ⚠️ ONE BACKUP PER RUN, NOT PER PASS. `merge_run` only takes one when it is going to
        # write, so `backup=not BACKED_UP` yields exactly one, taken before the first real
        # write — 70 timestamped copies of three CSVs answer "what did this change?" worse
        # than one (`pdf_ocr_job._upsert_period` takes the same line).
        _rep = pdf_ocr_merge.merge_run(
            LATEST, apply=MERGE_APPLY, periods=[_period], reports=MERGE_REPORTS,
            force_empty_band=FORCE_EMPTY_BAND,
            backup=not BACKED_UP, quiet=True)
        for _line in _rep.lines()[1:]:                # [0] repeats the ticker header
            print("  " + _line.strip())
        if _rep.backup:
            BACKED_UP = True
            print(f"  backup: {_rep.backup}")
        # ⚠️ THE ARTEFACT MUST RECORD WHAT THE MERGE DID, and on a KAGGLE run nothing else can:
        # `metadata.json` is written by the worker, which has no path to this disk (`MRG-1`).
        # `record_merge` APPENDS an event, which is why two passes do not overwrite each other.
        if MERGE_APPLY:
            pdf_ocr_merge.record_merge(LATEST, _rep)
        print()

    if not MERGE_APPLY:
        print("nothing was written. Set MERGE_APPLY = True to apply the plan above.")
        print("⚠️ AND THE PLAN ABOVE UNDERSTATES IT, BY CONSTRUCTION: pass 1 wrote no span, so")
        print("   pass 2 was planned against the blank one and reports the refusal it always")
        print("   would. A dry run cannot show a second pass that depends on the first.")
    else:
        # ⚠️ RE-READ FROM DISK so section 10 reports THIS merge rather than the state section 7
        # read before it ran. `MERGE` is what section 10 credits rows to.
        META = json.loads((LATEST / "metadata.json").read_text(encoding="utf-8"))
        MERGE = META.get("merge")
        print(f"-> {MERGE['statements_written']} statement(s) written, "
              f"{MERGE['statements_skipped']} refused"
              + (f"   quarters: {', '.join(MERGE['periods_written'])}"
                 if MERGE["periods_written"] else ""))


## 10 · Did it land? — the statement CSVs themselves

In [ ]:
# ── DID IT LAND? — the statement CSVs themselves ───────────────────────────
# ⚠️ Everything above reports what some process DECIDED; this reads what is on disk. The two
# came apart on a Kaggle round trip that finished green, wrote a complete run folder and created
# no CSV at all (`BND-1`, HOSE_BSR and again HOSE_CTG) — and no amount of log reading would have
# said so, because the merge that refused everything ran on the other machine.
import csv                                            # noqa: E402

from web_scraper import cafef_financials as fin       # noqa: E402

# ⚠️ `CWD-1`, AND THIS CELL WALKED STRAIGHT INTO IT. `statement_path()` reads
# `fin.STATEMENTS_DIR` at call time and its module default is RELATIVE — while the SETUP cell
# `os.chdir`s to `src/kaggle_gpu`, where `kgpu` stages its payload. So the first version of this
# cell reported `NO FILE` for a ticker whose three CSVs were on disk. The path is PRINTED,
# because a directory nobody names is a directory nobody checks.
ROOT = job.use_data_root(job.DEFAULT_DATA_ROOT)

TPL = (META or {}).get("inputs", {}).get("template") or TEMPLATE
# ⚠️ KEYED BY (period, REPORT), not by period. A merge writes one statement of a quarter and
# skips another — VCB Q1-2026 wrote its income statement and cash flow while its balance sheet
# was `identical to the row already on disk` — so a period-only set credits this run with a row
# it deliberately left alone.
MINE = {(d["period"], d["report"])
        for ev in (MERGE or {}).get("events", [])
        for d in ev["decisions"] if d["action"] == "write" and ev["applied"]}
if TPL is None:
    print("no template resolved — run the cells above first")
else:
    print(f"{ROOT / 'financials' / 'statements' / TPL}   {EXCHANGE}_{SYMBOL}")
    print()
    ANY = False
    for _report in fin.REPORTS:
        _path = Path(fin.statement_path(TPL, _report, EXCHANGE, SYMBOL))
        if not _path.is_file():
            print(f"  {_report:18} ⚠️ NO FILE — {_path.name} does not exist")
            continue
        ANY = True
        with open(_path, encoding="utf-8-sig") as _f:
            _rows = list(csv.DictReader(_f))
        _src = {}
        for _r in _rows:
            _src[_r.get("source", "")] = _src.get(_r.get("source", ""), 0) + 1
        _mine = [_r for _r in _rows if _r.get("source") == "pdf"
                 and (_r["period"], _report) in MINE]
        print(f"  {_report:18} {len(_rows):>3} quarters   "
              + "  ".join(f"{k}={v}" for k, v in sorted(_src.items()))
              + (f"   <- {len(_mine)} from this run" if _mine else ""))
        # ⚠️ Rule 24: a financial statement comes from the filing PDF and from nothing else.
        # A `cafef` row is an HTML transcription and must not be in this file.
        if _src.get("cafef"):
            print(f"       ⚠️ {_src['cafef']} row(s) read `source=cafef` — an HTML "
                  f"transcription. §5 rule 24 forbids it.")
    if not ANY:
        print()
        print("  ⚠️ THIS TICKER HAS NO STATEMENT CSV AT ALL. The parse is in the run folder "
              "and")
        print("     nothing was upserted — the MERGE section above says which refusal "
              "stopped it.")


## 11 · Repair one row — scoped, deliberate, read the diff first

In [ ]:
# ── REPAIR ONE ROW — scoped, deliberate, and read the diff first ──────────
# ⚠️ THE ONLY WAY THIS NOTEBOOK OVERWRITES A GOOD-LOOKING `pdf` ROW. `pdf_ocr_merge` refuses a
# figure that DIFFERS from a `pdf` row on disk, because two runs disagreeing is not settled by
# preferring the newer one. That refusal is lifted here for the NAMED pairs only — never for the
# run — and `merge_run`'s own `periods`/`reports` filter is what scopes it.
# ⚠️ A BACKUP is taken before any write. Diff EVERY COLUMN afterwards, not the figures: three
# separate runs in this repo lost only a `publish_date` and a figures-only diff called each of
# them clean (CLAUDE.md §6-2-quatervicies, §6-2-quinvicies, §6-2-quadragies).
if LATEST is not None and REPAIR:
    from web_scraper import pdf_ocr_merge                 # noqa: E402

    HOW = "APPLY" if REPAIR_APPLY else "PLAN"
    print(f"{HOW} — {len(REPAIR)} scoped repair(s) from {LATEST.name}")
    print()
    for _period, _report in REPAIR:
        print(f"── {_period} {_report} " + "─" * 46)
        _rep = pdf_ocr_merge.merge_run(
            LATEST, apply=REPAIR_APPLY, periods=[_period], reports=[_report],
            force_differs=True, force_empty_band=FORCE_EMPTY_BAND)
        if getattr(_rep, "backup", None):
            print(f"   backup: {_rep.backup}")
    if not REPAIR_APPLY:
        print()
        print("nothing was written. Set REPAIR_APPLY = True to apply the plan above.")
elif LATEST is not None:
    print("REPAIR is empty — no row already on disk was replaced.")
    print("  A statement this run parsed that disk already holds as `pdf` was refused as")
    print("  DIFFERS and left alone. That is the default and usually right; name the")
    print("  (quarter, statement) pair in REPAIR only once the FILING has settled which")
    print("  reading is correct.")
